<a href="https://colab.research.google.com/github/gaelosvaldor-star/Tercer-paracial-/blob/main/Variables_antit%C3%A9ticas_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Monte Carlo antitético con criterio de paro

Aproximamos $I=\int_{0}^{1} e^{x^2}\,dx$. Como $e^{x^2}$ no tiene primitiva elemental, usamos Monte Carlo con $f(x)=e^{x^2}$ y $U\sim\mathrm{Unif}(0,1)$, ya que $\mathbb{E}[f(U)]=I$.

Como $1-U$ también es $\mathrm{Unif}(0,1)$ y $f$ es **creciente**, $f(U)$ y $f(1-U)$ quedan negativamente correlacionadas, así que el promedio antitético

$$Y=\frac{f(U)+f(1-U)}{2}$$

tiene **menor varianza** que $f(U)$, sin más costo. En vez de fijar $N$, generamos muestras $Y_j$ y actualizamos media y varianza de forma recursiva:

$$\bar{Y}_j=\bar{Y}_{j-1}+\frac{Y_j-\bar{Y}_{j-1}}{j},\qquad S_j^2=\Big(1-\tfrac{1}{j}\Big)S_{j-1}^2+(j-1)\big(\bar{Y}_j-\bar{Y}_{j-1}\big)^2$$

Paramos cuando el error estándar $\sqrt{S_j^2/j}<d$. El valor de referencia es $I\approx 1.46265$.

In [9]:
import numpy as np
import random as r

def f(x):
    return np.exp(x**2)

# Monte Carlo antitético con criterio de paro.
# d : precisión deseada (cota para el error estándar de la media)
# n : número mínimo de iteraciones antes de revisar el criterio
def mc_anti(f, d, n=100):
    media = 0
    S = 0
    j = 0
    error = 0
    while True:
        U = r.random()
        x = (f(U) + f(1-U))/2                              # pareja antitética promediada
        j += 1
        if j == 1:
            media = x
            S = 0
        else:
            media_ant = media
            media = media_ant + (x - media_ant)/j          # media recursiva (Welford)
            S = (1 - 1/j)*S + (j - 1)*(media - media_ant)**2  # varianza recursiva
        if j >= n:
            error = np.sqrt(S/j)                           # error estándar de la media
            if error < d:
                break
    return media, error, np.sqrt(S), j

# --- RESULTADOS ---
media, error, desv, j = mc_anti(f, 0.001)
print("media  =", media)
print("error  =", error)
print("desv   =", desv)
print("iter j =", j)

media  = 1.4630013784165175
error  = 0.0009999819880249852
desv   = 0.1674036751027068
iter j = 28025
